# Instruction Fine-Tuning: TinyGPT on Dolly 15K

**Goal:** Take the pretrained TinyGPT base model and instruction fine-tune it
using the Databricks Dolly 15K dataset, using a custom PyTorch training loop.

**Setup required before running:**
1. Mount Google Drive (cell below)
2. Ensure `tinygpt.py` is at `/content/drive/MyDrive/tinygpt.py`
3. Ensure `tinygpt_weights.pt` is at `/content/drive/MyDrive/models/tinygpt_weights.pt`

## Step 1: Mount Drive and Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tiktoken datasets

## Step 2: Config

In [ ]:
import sys
import os
import torch
import torch.nn.functional as F
import tiktoken
import matplotlib.pyplot as plt
from datasets import load_dataset

# --- Paths ---
TINYGPT_DIR     = "/content/drive/MyDrive"              # where tinygpt.py lives
WEIGHTS_PATH    = "/content/drive/MyDrive/models/tinygpt_weights.pt"
FINETUNE_CKPT   = "/content/drive/MyDrive/models/tinygpt_dolly_ft.pt"  # output

# --- Fine-tuning Hyperparameters ---
BATCH_SIZE           = 4
EFFECTIVE_BATCH_SIZE = 64    # accumulation_steps = 64/4 = 16
MAX_STEPS            = 3000
LEARNING_RATE        = 1e-4  # lower than pretraining
WEIGHT_DECAY         = 0.01
GRAD_CLIP            = 1.0
WARMUP_STEPS         = 100
EVAL_STEPS           = 200   # evaluate every N steps
SAVE_STEPS           = 500   # save checkpoint every N steps
MAX_SEQ_LEN          = 512   # shorter than pretraining 1024 — Dolly examples are short
VAL_SPLIT            = 0.1

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

assert EFFECTIVE_BATCH_SIZE % BATCH_SIZE == 0
accumulation_steps = EFFECTIVE_BATCH_SIZE // BATCH_SIZE
print(f"Accumulation steps: {accumulation_steps}")

## Step 3: Load TinyGPT Model

In [ ]:
sys.path.insert(0, TINYGPT_DIR)
import tinygpt

# Load tokenizer
enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token  # 50256 — used as padding and end-of-text

# Load model in TRAIN mode (not eval)
state = tinygpt.State(
    tokenizer=enc,
    train_data=None,
    val_data=None,
    vocab_size=enc.n_vocab
)
model = tinygpt.TinyGPT(state).to(device)

# Load pretrained weights (PyTorch native format)
state_dict = torch.load(WEIGHTS_PATH, map_location=device, weights_only=True)
if any(k.startswith("_orig_mod.") for k in state_dict):
    state_dict = {k.removeprefix("_orig_mod."): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.train()  # switch to training mode

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded. Parameters: {total_params/1e6:.1f}M")

## Step 4: Load and Format Dolly 15K

The core of instruction fine-tuning — we define a prompt template that teaches
the model what an instruction looks like, what a response looks like, and where
it ends. The model learns this structure through repeated exposure.

In [ ]:
print("Loading Dolly 15K...")
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
print(f"Total examples: {len(dataset)}")
print(f"Sample: {dataset[0]}")

In [ ]:
def format_and_tokenize(example):
    """Format a Dolly example into instruction-response format and tokenize.

    Returns a list of token ids, or None if the example is too long.
    The EOT token marks the end of the response — critical for the model
    to learn where to stop generating.
    """
    instruction = example["instruction"].strip()
    context     = example["context"].strip()
    response    = example["response"].strip()

    if context:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Context:\n{context}\n\n"
            f"### Response:\n{response}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{response}"
        )

    tokens = enc.encode_ordinary(text)
    tokens.append(EOT)  # end of sequence marker

    if len(tokens) > MAX_SEQ_LEN:
        return None  # skip examples that are too long
    return tokens


print("Tokenizing dataset...")
all_tokens = []
skipped = 0
for ex in dataset:
    tokens = format_and_tokenize(ex)
    if tokens is None:
        skipped += 1
        continue
    all_tokens.append(tokens)

print(f"Kept: {len(all_tokens)} examples | Skipped (too long): {skipped}")

# Train / val split
split_idx  = int(len(all_tokens) * (1 - VAL_SPLIT))
train_data = all_tokens[:split_idx]
val_data   = all_tokens[split_idx:]
print(f"Train: {len(train_data)} | Val: {len(val_data)}")

# Preview formatted example
print("\nFormatted example (decoded):")
print(enc.decode(train_data[0]))

## Step 5: Batch Sampler

Since Dolly examples have variable lengths, we pad shorter sequences to
the longest in the batch. Padding tokens are masked out in the loss
so they don't affect training.

In [ ]:
import random

def get_batch(data, batch_size):
    """Sample a random batch, pad to equal length, return (x, y, mask)."""
    batch = random.sample(data, batch_size)

    max_len = max(len(s) for s in batch)

    x_list, y_list, mask_list = [], [], []
    for tokens in batch:
        pad_len = max_len - len(tokens)
        # x: input tokens (all but last)
        # y: target tokens (all but first, shifted by 1)
        padded = tokens + [EOT] * pad_len
        x = padded[:-1]
        y = padded[1:]
        # mask: 1 where tokens are real, 0 where padded
        mask = [1] * (len(tokens) - 1) + [0] * pad_len
        x_list.append(x)
        y_list.append(y)
        mask_list.append(mask)

    x    = torch.tensor(x_list, dtype=torch.long).to(device)
    y    = torch.tensor(y_list, dtype=torch.long).to(device)
    mask = torch.tensor(mask_list, dtype=torch.float).to(device)
    return x, y, mask


def compute_loss(model, x, y, mask):
    """Forward pass + masked cross entropy loss."""
    logits = model(x)                          # (B, T, vocab)
    B, T, C = logits.shape
    loss = F.cross_entropy(
        logits.view(B * T, C),
        y.view(B * T),
        reduction="none"
    )                                          # (B*T,)
    loss = loss * mask.view(B * T)             # zero out padding positions
    return loss.sum() / mask.sum()             # mean over real tokens only


@torch.no_grad()
def evaluate(model, data, num_batches=20):
    """Average loss over num_batches random val batches."""
    model.eval()
    losses = []
    for _ in range(num_batches):
        x, y, mask = get_batch(data, BATCH_SIZE)
        losses.append(compute_loss(model, x, y, mask).item())
    model.train()
    return sum(losses) / len(losses)

print("Batch sampler ready.")

## Step 6: Baseline — Test BEFORE Fine-Tuning

In [ ]:
test_prompts = [
    "What is photosynthesis?",
    "Explain the water cycle in simple terms.",
    "Who was the first emperor of Rome?",
]

def test_model(model, prompts, label=""):
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")
    model.eval()
    for prompt in prompts:
        full_prompt = f"### Instruction:\n{prompt}\n\n### Response:\n"
        print(f"\nQ: {prompt}")
        print(f"A: {model.generate_text(start_text=full_prompt, max_tokens=150, temperature=0.7)}")
        print("-" * 40)
    model.train()

test_model(model, test_prompts, label="BASELINE (Before Fine-Tuning)")

## Step 7: Optimizer and Scheduler

In [ ]:
# Separate decay and no-decay params (same pattern as pretraining)
decay_params   = [p for n, p in model.named_parameters() if p.dim() >= 2]
nodecay_params = [p for n, p in model.named_parameters() if p.dim() < 2]
print(f"Decay params:   {sum(p.numel() for p in decay_params)/1e6:.1f}M")
print(f"No-decay params:{sum(p.numel() for p in nodecay_params)/1e6:.1f}M")

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params,   "weight_decay": WEIGHT_DECAY},
        {"params": nodecay_params, "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    eps=1e-8,
)

# Linear warmup then cosine decay
scheduler_warmup = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_STEPS
)
scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=(MAX_STEPS - WARMUP_STEPS), eta_min=LEARNING_RATE * 0.1
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[scheduler_warmup, scheduler_cosine],
    milestones=[WARMUP_STEPS]
)

print("Optimizer and scheduler ready.")

## Step 8: Resume from Checkpoint (Optional)

If resuming a previous fine-tuning run, set `RESUME_CKPT` to the checkpoint
file path and run this cell. Skip if starting fresh.

In [ ]:
# Set to None to start fresh, or path to resume e.g.:
# RESUME_CKPT = "/content/drive/MyDrive/models/tinygpt_dolly_step1000.pt"
RESUME_CKPT = None

start_step = 0

if RESUME_CKPT and os.path.exists(RESUME_CKPT):
    print(f"Resuming from: {RESUME_CKPT}")
    ckpt = torch.load(RESUME_CKPT, map_location=device, weights_only=False)
    # Load model weights
    state_dict = ckpt["model_state"]
    if any(k.startswith("_orig_mod.") for k in state_dict):
        state_dict = {k.removeprefix("_orig_mod."): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    # Restore optimizer and scheduler
    optimizer.load_state_dict(ckpt["optimizer_state"])
    if ckpt.get("scheduler_state") is not None:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    start_step = int(ckpt.get("step", 0)) + 1
    print(f"Resumed at step {start_step} — training will run from step {start_step} to {MAX_STEPS}")
else:
    print("Starting fresh from step 0.")

## Step 9: Training Loop

In [ ]:
steps_log       = []
train_loss_log  = []
val_loss_log    = []

print(f"Starting fine-tuning from step {start_step} to {MAX_STEPS}...")
print(f"Effective batch size: {EFFECTIVE_BATCH_SIZE} ({accumulation_steps} accumulation steps)")
print(f"Eval every {EVAL_STEPS} steps | Save every {SAVE_STEPS} steps")
print("=" * 60)

model.train()

for step in range(start_step, MAX_STEPS):

    # --- Gradient Accumulation ---
    optimizer.zero_grad(set_to_none=True)
    micro_loss_total = 0.0

    for _ in range(accumulation_steps):
        x, y, mask = get_batch(train_data, BATCH_SIZE)
        if device == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                loss = compute_loss(model, x, y, mask)
        else:
            loss = compute_loss(model, x, y, mask)
        scaled_loss = loss / accumulation_steps
        scaled_loss.backward()
        micro_loss_total += loss.item()

    avg_train_loss = micro_loss_total / accumulation_steps

    # --- Clip and Update ---
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()
    scheduler.step()

    # --- Logging ---
    if (step + 1) % EVAL_STEPS == 0 or step == 0:
        val_loss = evaluate(model, val_data)
        lr_now   = scheduler.get_last_lr()[0]
        steps_log.append(step + 1)
        train_loss_log.append(avg_train_loss)
        val_loss_log.append(val_loss)
        print(f"step {step+1:>5} | train {avg_train_loss:.4f} | val {val_loss:.4f} | lr {lr_now:.2e}")

    # --- Checkpoint ---
    if (step + 1) % SAVE_STEPS == 0:
        ckpt_path = f"/content/drive/MyDrive/models/tinygpt_dolly_step{step+1}.pt"
        torch.save({
            "step":            step,
            "model_state":     model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
        }, ckpt_path)
        print(f"  → Checkpoint saved: {ckpt_path}")

print("\nTraining complete!")

## Step 10: Plot Loss Curves

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(steps_log, train_loss_log, label="Train Loss", linewidth=2)
plt.plot(steps_log, val_loss_log,   label="Val Loss",   linewidth=2, marker="o", markersize=5)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Fine-Tuning Loss — TinyGPT on Dolly 15K")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/models/finetune_loss_curve.png", dpi=150)
plt.show()
print("Loss curve saved to Drive.")

## Step 11: Test AFTER Fine-Tuning

Same prompts as baseline. Compare the delta — that's your intuition moment.

In [ ]:
test_model(model, test_prompts, label="AFTER Instruction Fine-Tuning")

## Step 12: Save Final Weights

In [ ]:
# Save inference-only weights (no optimizer state)
final_weights_path = "/content/drive/MyDrive/models/tinygpt_dolly_weights.pt"
state_dict = model.state_dict()
torch.save(state_dict, final_weights_path)

size_mb = os.path.getsize(final_weights_path) / 1024 / 1024
print(f"Final weights saved: {final_weights_path} ({size_mb:.0f} MB)")

## What Just Happened?

| Stage | What the model learned |
|-------|------------------------|
| Pretraining (base model) | Predict next tokens from raw FineWeb-Edu text |
| Instruction fine-tuning (this notebook) | Recognize `### Instruction / ### Response` structure and generate helpful answers |

**Next step:** Take this instruction-tuned TinyGPT and do domain SFT on
Indian mythology Q&A pairs — or repeat this pipeline on Qwen3-4B with LoRA
for a production-grade result.